In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing
import seaborn as sns
from sklearn .preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

C:\Users\mahen\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
data = pd.read_csv(r'D:\Data science\FSDS 4pm\car-mpg.csv')

In [3]:
data

,mpg,cyl,disp,hp,wt,acc,yr,origin,car_type,car_name
0,18.0,8,307.0,130,3504,12.0,70,1,0,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,0,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,0,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,0,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,0,ford torino
...,...,...,...,...,...,...,...,...,...,...
393,27.0,4,140.0,86,2790,15.6,82,1,1,ford mustang gl
394,44.0,4,97.0,52,2130,24.6,82,2,1,vw pickup
395,32.0,4,135.0,84,2295,11.6,82,1,1,dodge rampage
396,28.0,4,120.0,79,2625,18.6,82,1,1,ford ranger


In [4]:
data = data.drop(['car_name'], axis =1)
data['origin'] = data['origin'].replace({1: 'america', 2: 'europe', 3: 'asia'})
data = pd.get_dummies(data,columns = ['origin'], dtype=int)
data = data.replace('?', np.nan)

In [5]:
data = data.apply(pd.to_numeric, errors='ignore')

#fill missing values with median only for numeric columns
numeric_cols = data.select_dtypes(include = [np.number]).columns
data[numeric_cols] = data[numeric_cols].apply(lambda x: x.fillna(x.median()))

C:\Users\mahen\AppData\Local\Temp\ipykernel_5888\971296043.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  data = data.apply(pd.to_numeric, errors='ignore')


In [6]:
data.head()

,mpg,cyl,disp,hp,wt,acc,yr,car_type,origin_america,origin_asia,origin_europe
0,18.0,8,307.0,130.0,3504,12.0,70,0,1,0,0
1,15.0,8,350.0,165.0,3693,11.5,70,0,1,0,0
2,18.0,8,318.0,150.0,3436,11.0,70,0,1,0,0
3,16.0,8,304.0,150.0,3433,12.0,70,0,1,0,0
4,17.0,8,302.0,140.0,3449,10.5,70,0,1,0,0


In [7]:
X = data.drop(['mpg'], axis = 1) # independent variable
y = data[['mpg']] #dependent variable

In [8]:
X_s = preprocessing.scale(X)
X_s = pd.DataFrame(X_s, columns = X.columns) #converting scaled data into dataframe

y_s = preprocessing.scale(y)
y_s = pd.DataFrame(y_s, columns = y.columns) #ideally train, test data should be in columns

In [9]:
#Split into train, test set

X_train, X_test, y_train,y_test = train_test_split(X_s, y_s, test_size = 0.30, random_state = 1)
X_train.shape

(278, 10)

# Simple Linear Model

In [10]:
#Fit simple linear model and find coefficients
regression_model = LinearRegression()
regression_model.fit(X_train, y_train)

for idx, col_name in enumerate(X_train.columns):
    print('The coefficient for{} is {}'.format(col_name, regression_model.coef_[0][idx]))
    
intercept = regression_model.intercept_[0]
print('The intercept is {}'.format(intercept))

The coefficient forcyl is 0.3210223856916108
The coefficient fordisp is 0.3248343091848394
The coefficient forhp is -0.2291695005943759
The coefficient forwt is -0.7112101905072299
The coefficient foracc is 0.014713682764191435
The coefficient foryr is 0.3755811949510741
The coefficient forcar_type is 0.38147694842331
The coefficient fororigin_america is -0.0747224754758417
The coefficient fororigin_asia is 0.04451525203567813
The coefficient fororigin_europe is 0.04834854953945371
The intercept is 0.019284116103639715


# Ridge Regression

In [11]:
ridge_model = Ridge(alpha = 0.4)
ridge_model.fit(X_train, y_train)

print('Ridge model coef: {}'.format(ridge_model.coef_))

Ridge model coef: [[ 0.31495967  0.30948411 -0.22861679 -0.69782283  0.01239531  0.37411266
   0.37586629 -0.07408168  0.04437854  0.0476772 ]]


# Lasso Regression

In [14]:
lasso_model = Lasso(alpha = 0.4)
lasso_model.fit(X_train, y_train)

print('Lasso model coef: {}'.format(lasso_model.coef_))

Lasso model coef: [-0.         -0.         -0.         -0.39425009  0.          0.05916007
  0.02075746 -0.          0.          0.        ]


# Score Comparison

In [15]:
#Simple Linear Model
print(regression_model.score(X_train, y_train))#Bias
print(regression_model.score(X_test, y_test))#Variance

0.8343770256960538
0.8513421387780067


In [17]:
#Ridge
print(ridge_model.score(X_train, y_train))
print(ridge_model.score(X_test, y_test))

0.8343502868181134
0.8520594956782537


In [18]:
#Lasso
print(lasso_model.score(X_train, y_train))
print(lasso_model.score(X_test, y_test))

0.556352624509626
0.5965524844000425
